In [1]:
import pandas as pd
import ast

# 1. Load the raw data
df = pd.read_csv('master_academic_data.csv')

print("Original shape:", df.shape)

# 2. Data Reduction: Drop metadata and personal identifiers (except PRN for tracking)
columns_to_drop = [
    '__v_x', '__v_y', 'createdAt_x', 'createdAt_y', 
    'updatedAt_x', 'updatedAt_y', 'batch', 'motherName', 'name', 'seatNo'
]
df_cleaned = df.drop(columns=columns_to_drop)

print("Shape after dropping noise:", df_cleaned.shape)
df_cleaned.head(3)

Original shape: (373, 20)
Shape after dropping noise: (373, 10)


,prn,category,gender,semester,finalResult,isKT,isNEP,sgpi,subjects,totalMarks
0,2022016402188296,Regular,Female,8,P,False,False,8.80,"{'ABCID': '=""784286047150""', 'Is_Diploma_Stude...",0
1,2023016402547974,Regular,Male,8,P,False,False,8.40,"{'ABCID': '=""350549767761""', 'Is_Diploma_Stude...",0
2,2022016402188466,Regular,Female,8,P,False,False,9.30,"{'ABCID': '=""699656948270""', 'Is_Diploma_Stude...",0


In [2]:
# Convert 'isKT' (True/False) to 1 and 0 (1 = Has a KT, 0 = Clear)
df_cleaned['isKT'] = df_cleaned['isKT'].astype(int)

# Convert Gender (Optional but sometimes useful for demographic clustering)
df_cleaned['gender'] = df_cleaned['gender'].map({'Male': 0, 'Female': 1})

# Clean the SGPI column (sometimes parsers capture these as strings or with errors)
# This forces everything to a number, turning errors into NaN (Not a Number)
df_cleaned['sgpi'] = pd.to_numeric(df_cleaned['sgpi'], errors='coerce')

# Fill any missing SGPIs with 0 (assuming missing means they failed or dropped)
df_cleaned['sgpi'] = df_cleaned['sgpi'].fillna(0)

print(df_cleaned[['prn', 'semester', 'sgpi', 'isKT']].head())


                prn  semester  sgpi  isKT
0  2022016402188296         8  8.80     0
1  2023016402547974         8  8.40     0
2  2022016402188466         8  9.30     0
3  2021016402129583         8  7.45     0
4  2021016402129583         1  9.67     0


In [3]:
# Function to safely convert the string representation of a dictionary into an actual dictionary
def parse_subjects(subject_string):
    try:
        # ast.literal_eval safely parses strings containing Python data structures
        return ast.literal_eval(subject_string)
    except:
        return {}

# Apply the function to the subjects column
df_cleaned['subjects_dict'] = df_cleaned['subjects'].apply(parse_subjects)

# Flatten the dictionaries into their own columns
# This will create a massive, wide dataframe where every subject component has its own column
subjects_df = pd.json_normalize(df_cleaned['subjects_dict'])

# Merge these new subject columns back into our cleaned dataframe
final_df = pd.concat([df_cleaned.drop(columns=['subjects', 'subjects_dict']), subjects_df], axis=1)

print("Final shape after unpacking subjects:", final_df.shape)

# Save this transformed data to a new CSV for the modeling phase!
final_df.to_csv('cleaned_academic_data.csv', index=False)
print("Data successfully cleaned and saved to cleaned_academic_data.csv")

Final shape after unpacking subjects: (373, 241)
Data successfully cleaned and saved to cleaned_academic_data.csv


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

# 1. Load the cleaned data
df = pd.read_csv('cleaned_academic_data.csv')

# 2. Define our Target (y) and Features (X)
y = df['isKT']

# Drop columns that are non-numeric or cause "data leakage" 
# (Data leakage is when you give the model the answer key, like the 'finalResult' string)
cols_to_drop = ['prn', 'isKT', 'finalResult', 'ABCID', 'Is_Diploma_Student', 'category']
X = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# Keep only numeric columns for this first baseline model
X = X.select_dtypes(include=['number'])

# Fill any remaining NaN (missing marks) with 0
X = X.fillna(0)

# 3. Split the data into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training on {X_train.shape[0]} students, Testing on {X_test.shape[0]} students...\n")

# 4. Initialize and Train the Machine Learning Model
model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
model.fit(X_train, y_train)

# 5. Make predictions on the unseen test data
y_pred = model.predict(X_test)

# 6. Evaluate the results
print("--- Model Accuracy ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")

print("--- Classification Report ---")
# 0 = No KT (Clear), 1 = KT
print(classification_report(y_test, y_pred))

Training on 298 students, Testing on 75 students...

--- Model Accuracy ---
Accuracy Score: 100.00%

--- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        75

    accuracy                           1.00        75
   macro avg       1.00      1.00      1.00        75
weighted avg       1.00      1.00      1.00        75



In [3]:
# Extract feature importances
importances = pd.DataFrame(
    model.feature_importances_, 
    index=X.columns, 
    columns=['Importance']
)

print("--- Top 10 Factors Predicting a KT ---")
print(importances.sort_values(by='Importance', ascending=False).head(10))

--- Top 10 Factors Predicting a KT ---
                              Importance
gender                               0.0
semester                             0.0
sgpi                                 0.0
BIG DATA ANALYTICS_Total             0.0
BLOCKCHAIN AND DLT_Total             0.0
BLOCKCHAIN LAB_Total                 0.0
CLOUD COMPUTING LAB_Total            0.0
ERP_Total                            0.0
KNOWLEDGE MANAGEMENT_In_PrOr         0.0
KNOWLEDGE MANAGEMENT_Th_Tw           0.0


In [6]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

# 1. Load the cleaned data
df = pd.read_csv('cleaned_academic_data.csv')

# 2. Fix the Target (y) 
# Derive true KT status: Student failed if finalResult is Unsuccessful/Fail, OR if SGPI is 0
df['Target_KT'] = df['finalResult'].astype(str).str.upper().isin(['UNSUCCESSFUL', 'F', 'FAIL']) | (df['sgpi'] == 0.0)
y = df['Target_KT'].astype(int)

# 3. Drop non-predictive metadata
# 3. Drop non-predictive metadata AND Data Leakage columns
cols_to_drop = [
    'prn', 'isKT', 'finalResult', 'ABCID', 'Is_Diploma_Student', 'category', 'Target_KT',
    'sgpi', 'Total Marks', 'totalMarks', 'semester' # <--- ADD THESE CHEAT COLUMNS HERE
]
X = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# 4. Clean the Subject Marks (Smarter extraction)
def clean_marks(val):
    if pd.isna(val):
        return 0.0
    
    val_str = str(val)
    # Search for the first valid integer or decimal number in the string
    match = re.search(r'\d+(\.\d+)?', val_str)
    
    if match:
        return float(match.group(0))
    return 0.0

# Apply the cleaner to every column except our known core numeric ones
for col in X.columns:
    if col not in ['gender', 'semester', 'sgpi']:
        X[col] = X[col].apply(clean_marks)

# Keep only the numeric columns 
X = X.select_dtypes(include=['number'])
X = X.fillna(0)

# 5. Train the Model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training on {X_train.shape[0]} students, Testing on {X_test.shape[0]} students...")
print(f"Actual KTs found in dataset: {y.sum()}\n")

# Initialize and train
model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print("--- Model Accuracy ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")

print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

# 6. Reveal the Bottlenecks
importances = pd.DataFrame(
    model.feature_importances_, 
    index=X.columns, 
    columns=['Importance']
)
print("--- Top 10 Factors Predicting a KT ---")
print(importances.sort_values(by='Importance', ascending=False).head(10))

# Save the working model
joblib.dump(model, 'kt_predictor_model.pkl')

Training on 298 students, Testing on 75 students...
Actual KTs found in dataset: 179

--- Model Accuracy ---
Accuracy Score: 97.33%

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.95      1.00      0.98        41
           1       1.00      0.94      0.97        34

    accuracy                           0.97        75
   macro avg       0.98      0.97      0.97        75
weighted avg       0.97      0.97      0.97        75

--- Top 10 Factors Predicting a KT ---
                               Importance
SECURITY LAB - TW (Marks)        0.046203
COA - IA (Marks)                 0.045554
CNND - ESE (Marks)               0.042642
IP LAB - TW (Marks)              0.041321
COA - TOT (Marks)                0.040603
ENGG MATHS - IV - TOT (Marks)    0.036496
OS - IA (Marks)                  0.030020
ENGG MATHS - IV - ESE (Marks)    0.028444
DevOPs LAB - PR OR (Marks)       0.025640
AT - ESE (Marks)                 0.022296


['kt_predictor_model.pkl']